In [104]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from statistics import mean


def parse_estimate(ls):
    value = float(ls.split("= ")[1].split(" ", 1)[0].strip())
    lo = float(ls.split("(")[1].split("-")[0].strip())
    hi = float(ls.split("-")[1].split(")")[0].strip())
    return value, lo, hi

In [105]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory (where figure will be saved): {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

Current working directory (where figure will be saved): /data/users/bdupin/datasets-organelle-igr/code
Folder where groups (data) folder should be: /data/users/bdupin/datasets-organelle-igr


In [106]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "green_algae_mit",
    "green_algae_plt",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
]

polarity_3bin = {
    "++": "same",
    "--": "same",
    "+-": "convergent",
    "-+": "divergent",
}

group_label1 = {
    "fungi_mit": "Fungi",
    "metazoans_mit": "Metazoans",
    "plants_mit": "Plants",
    "protists_mit": "Protists",
    "green_algae_mit": "Green algae",
    "plants_plt": "Plants ",
    "protists_plt": "Protists ",
    "green_algae_plt": "Green algae ",
}

organelle_dict = {
    "Fungi": "Mitochondria",
    "Metazoans": "Mitochondria",
    "Plants": "Mitochondria",
    "Protists": "Mitochondria",
    "Green algae": "Mitochondria",
    "Plants ": "Plastid",
    "Protists ": "Plastid",
    "Green algae ": "Plastid",
}

group_label2 = {
    "fungi_mit": "Fungi (mitochondria)",
    "metazoans_mit": "Metazoans (mitochondria)",
    "green_algae_mit": "Green algae (mitochondria)",
    "green_algae_plt": "Green algae (plastids)",
    "plants_mit": "Plants (mitochondria)",
    "plants_plt": "Plants (plastids)",
    "protists_mit": "Protists (mitochondria)",
    "protists_plt": "Protists (plastids)",
}

GROUP_ORDER = [
    "Fungi",
    "Metazoans",
    "Plants",
    "Protists",
    "Green algae",
    "Plants ",
    "Protists ",
    "Green algae ",
]

MIT_GROUPS = {"Fungi", "Metazoans", "Plants", "Protists", "Green algae"}

COLOR_SAME = "#D55E00"  # vermillion
COLOR_CONV = "#56B4E9"  # sky blue
COLOR_DIV = "#03045E"  # deep navy

In [107]:
rows = []
medians_3bin = {}

for g in groups:
    igs = pl.read_csv(BASE / g / "summary_igs_intergenic.tsv", separator="\t")
    brm_result = BASE / g / "brms_3bin" / "brms_result.txt"

    igs = igs.with_columns(
        pl.col("Polarity").replace_strict(polarity_3bin).alias("polarity_3bin")
    )
    med = igs.group_by(["AN", "polarity_3bin"]).agg(pl.col("Length").median())
    med = med.pivot(values="Length", index="AN", on="polarity_3bin")
    medians_3bin[g] = med

    for line in brm_result.read_text().splitlines():
        ls = line.strip()
        if "Delta PGLMM b_polarity_binconvergent" in ls:
            conv_b, conv_lo, conv_hi = parse_estimate(ls)
        elif "Fold  PGLMM b_polarity_binconvergent" in ls:
            conv_fold, conv_fold_lo, conv_fold_hi = parse_estimate(ls)
        elif "Delta PGLMM b_polarity_bindivergent" in ls:
            div_b, div_lo, div_hi = parse_estimate(ls)
        elif "Fold  PGLMM b_polarity_bindivergent" in ls:
            div_fold, div_fold_lo, div_fold_hi = parse_estimate(ls)

    label = group_label1.get(g, g)
    organelle = organelle_dict.get(label, "Unknown")
    rows.append(
        {
            "group": label,
            "polarity": "convergent",
            "real_group": g,
            "fold": conv_fold,
            "ci_low": conv_fold_lo,
            "ci_high": conv_fold_hi,
            "organelle": organelle,
        }
    )
    rows.append(
        {
            "group": label,
            "polarity": "divergent",
            "real_group": g,
            "fold": div_fold,
            "ci_low": div_fold_lo,
            "ci_high": div_fold_hi,
            "organelle": organelle,
        }
    )

    print(f"{g}")
    print(
        f"  Medians: Same={med['same'].median():.1f}, Conv={med['convergent'].median():.1f}, Div={med['divergent'].median():.1f}"
    )
    print(
        f"  Conv PGLMM={conv_b:.3f} ({conv_lo:.3f},{conv_hi:.3f}), Fold={conv_fold:.3f} ({conv_fold_lo:.3f},{conv_fold_hi:.3f})"
    )
    print(
        f"  Div  PGLMM={div_b:.3f} ({div_lo:.3f},{div_hi:.3f}), Fold={div_fold:.3f} ({div_fold_lo:.3f},{div_fold_hi:.3f})\n"
    )

data = pl.DataFrame(rows)
n_by_label = {group_label1[g]: med.shape[0] for g, med in medians_3bin.items()}

fungi_mit
  Medians: Same=227.0, Conv=915.0, Div=614.0
  Conv PGLMM=0.507 (0.462,0.553), Fold=3.216 (2.896,3.571)
  Div  PGLMM=0.443 (0.398,0.487), Fold=2.773 (2.501,3.069)

metazoans_mit
  Medians: Same=3.5, Conv=9.0, Div=4.0
  Conv PGLMM=0.209 (0.200,0.218), Fold=1.616 (1.584,1.650)
  Div  PGLMM=0.460 (0.453,0.467), Fold=2.884 (2.838,2.931)

green_algae_mit
  Medians: Same=172.0, Conv=446.0, Div=268.5
  Conv PGLMM=0.244 (0.178,0.310), Fold=1.752 (1.508,2.039)
  Div  PGLMM=0.143 (0.074,0.211), Fold=1.388 (1.186,1.624)

green_algae_plt
  Medians: Same=314.5, Conv=324.5, Div=467.0
  Conv PGLMM=0.120 (0.099,0.141), Fold=1.319 (1.257,1.382)
  Div  PGLMM=0.325 (0.304,0.347), Fold=2.116 (2.013,2.221)

plants_mit
  Medians: Same=1572.0, Conv=6795.5, Div=6004.5
  Conv PGLMM=0.544 (0.526,0.561), Fold=3.497 (3.354,3.643)
  Div  PGLMM=0.501 (0.484,0.519), Fold=3.169 (3.047,3.300)

plants_plt
  Medians: Same=200.5, Conv=245.0, Div=516.0
  Conv PGLMM=0.161 (0.159,0.163), Fold=1.448 (1.442,1.454)
 

In [108]:
# Composite figure layout
width = 13
forest_height = width / 2
boxplot_height = 17
dpi = 300

fig = plt.figure(figsize=(width, forest_height + boxplot_height), dpi=dpi)
gs = GridSpec(
    nrows=2,
    ncols=1,
    height_ratios=[forest_height, boxplot_height],
    hspace=0.2,
    figure=fig,
)

<Figure size 3900x7050 with 0 Axes>

In [109]:
# ==========
# Panel A: forest plot
# ==========
axA = fig.add_subplot(gs[0, 0])

positions = list(range(1, len(GROUP_ORDER) + 1))
x_pos = {g: i for i, g in zip(positions, GROUP_ORDER)}
small_marker = {"Metazoans", "Plants ", "Protists "}

axA.axhline(1.0, linewidth=1)

for row in data.iter_rows(named=True):
    xi = x_pos[row["group"]]
    is_conv = row["polarity"] == "convergent"
    x = xi - 0.15 if is_conv else xi + 0.15
    col = COLOR_CONV if is_conv else COLOR_DIV
    axA.errorbar(
        x,
        row["fold"],
        yerr=[[row["fold"] - row["ci_low"]], [row["ci_high"] - row["fold"]]],
        fmt="o",
        color=col,
        markersize=7 if row["group"] not in small_marker else 3,
        capsize=4,
        capthick=1.5,
        elinewidth=1.5,
        zorder=3,
    )

axA.axvline(5.5, color="gray", linestyle="--", alpha=0.3, linewidth=1.5)
axA.set_xticks(positions)
axA.set_xticklabels(
    [f"{g.strip()}\n(n={n_by_label.get(g.strip(), '')})" for g in GROUP_ORDER],
    rotation=0,
    fontsize=13,
)
axA.set_xlim(0.5, len(GROUP_ORDER) + 0.5)
axA.set_ylabel("Fold change\n(PGLMM, 95% CI)", fontsize=13)
axA.grid(axis="y", alpha=0.2)
axA.legend(
    handles=[
        mpatches.Patch(color=COLOR_CONV, label="Convergent"),
        mpatches.Patch(color=COLOR_DIV, label="Divergent"),
    ],
    frameon=False,
    fontsize=11.5,
    loc="upper right",
)

mit_positions = positions[:5]
plt_positions = positions[5:]
for xc, label in [
    (mean(mit_positions), "Mitochondria"),
    (mean(plt_positions), "Plastid"),
]:
    axA.text(
        xc,
        -0.15,
        label,
        ha="center",
        va="top",
        fontweight="bold",
        fontsize=12,
        transform=axA.get_xaxis_transform(),
    )

fig.text(0.125, 0.895, "A", fontsize=16, fontweight="bold", va="top")

Text(0.125, 0.895, 'A')

In [110]:
# ==========
# Panel B: 4x2 grid of boxplots
# ==========
subgs = gs[1, 0].subgridspec(4, 2, hspace=0.35, wspace=0.25)

for idx, g in enumerate(groups):
    r = idx // 2
    c = idx % 2
    ax = fig.add_subplot(subgs[r, c])

    med = medians_3bin[g]
    box_data = [
        med["convergent"],
        med["divergent"],
        med["same"],
    ]

    bp = ax.boxplot(
        box_data,
        tick_labels=["Same", "Convergent", "Divergent"],
        patch_artist=True,
        flierprops=dict(marker="o", markersize=1, alpha=0.3),
        medianprops=dict(linewidth=2, color="black"),
    )

    for patch, color in zip(bp["boxes"], [COLOR_SAME, COLOR_CONV, COLOR_DIV]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(group_label2[g], fontsize=14)
    ax.tick_params(axis="both", labelsize=13)
    ax.set_yscale("log")
    ax.set_ylabel(r"$\mathrm{median}(\mathrm{IGR\ length})$" "\n(bp, log)", fontsize=13)
    ax.set_xlabel(
        "" if r < 3 else "Gene-pair polarity type", fontsize=12, fontweight="bold"
    )

fig.text(0.125, 0.63, "B", fontsize=16, fontweight="bold", va="top")

plt.subplots_adjust(bottom=0.18)

<Figure size 640x480 with 0 Axes>

In [111]:
out_path = CWD / "figure2.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved composite figure to: {out_path}")

Saved composite figure to: /data/users/bdupin/datasets-organelle-igr/code/figure2.png
